# Merging canvases

Normally each person runs their own `Canvas` on their own port. This notebook
launches two such canvases (as if they were two different people / machines)
and then a **merge host** that composites both onto one new port — the union of
every panel, on a single surface.

The key property: interacting with a panel in the merged view runs that panel's
logic back in the process that created it. Here both source canvases live in
this one kernel for demonstration, but they could just as easily be on two
laptops — the merge host only needs to reach their ports.

Because a notebook kernel stays alive, the source canvases keep serving their
panels for as long as this kernel runs — which is exactly what the merge host
needs to route interactions back to them.

In [7]:
import pycanvas

## "Sarah's" canvas, on port 8001

Give the panels explicit positions. A position-less panel has no coordinate to
preserve (each source's browser auto-places it, and that spot is never reported
back), so the merged view would just auto-cascade it. With explicit `x`/`y`,
each panel appears in the merged view exactly where its owner placed it.

In [8]:
sarah = pycanvas.Canvas()
s_slider = sarah.insert(
    pycanvas.Slider(label="sarah_dial", min=0, max=100, default=20), x=120, y=240
)


@s_slider.on_change
def on_sarah(value):
    print(f"[sarah's process] dial -> {value}")


sarah.serve_background(port=8001, open_browser=True)

## "Josef's" canvas, on port 8002

In [9]:
josef = pycanvas.Canvas()
j_slider = josef.insert(
    pycanvas.Slider(label="josef_dial", min=0, max=100, default=80), x=120, y=100
)


@j_slider.on_change
def on_josef(value):
    print(f"[josef's process] dial -> {value}")


josef.serve_background(port=8002, open_browser=True)

## The merge host: unify both onto port 8080

By default the canvases are overlaid with their real coordinates preserved.
Pass `region_width=...` (e.g. `pycanvas.Merge([8001, 8002], region_width=1500)`)
to instead spread each source into its own side-by-side region.

`serve_background` returns immediately so the kernel stays interactive. Open the
printed URL: you'll see **both** sliders on one canvas. Drag either one — its
`on_change` fires in its own canvas (watch the printed `[sarah's process]` /
`[josef's process]` lines).

In [10]:
merged = pycanvas.Merge([8001, 8002]).serve_background(port=8080, open_browser=True)
print("Merged canvas at http://127.0.0.1:8080")

Merged canvas at http://127.0.0.1:8080


[merge] connected to localhost:8002
[merge] connected to localhost:8001


## Drive a panel from Python

Each source is still a normal `Canvas` you can control from this kernel. Setting
a slider's value in Python pushes to every view — including the merged one.

In [11]:
s_slider.update(75)   # moves sarah_dial in both Sarah's canvas and the merged view

## Shut everything down

Stop the merge host and both source canvases.

In [12]:
# merged.stop()
# sarah.stop()
# josef.stop()